<a href="https://colab.research.google.com/github/Arunodaya9027/Sales-Store-Time-Series/blob/main/TimeSeries.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q tsfresh "autogluon.timeseries[all]"

In [2]:
import pandas as pd
import numpy as np
from tqdm import tqdm
# from autogluon.timeseries import TimeSeriesPredictor
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import seasonal_decompose


# Load data
#train = pd.read_csv('/content/kaggle/input/store-sales-time-series-forecasting/train.csv')
# test = pd.read_csv('/content/kaggle/input/store-sales-time-series-forecasting/test.csv')


# Feature Engineering

In [3]:
# Load only necessary columns and downcast data types
oil_data = pd.read_csv('/content/kaggle/input/store-sales-time-series-forecasting/oil.csv',
                       usecols=['date', 'dcoilwtico'])
holidays_data = pd.read_csv('/content/kaggle/input/store-sales-time-series-forecasting/holidays_events.csv',
                            usecols=['date', 'type', 'locale'])
transactions_data = pd.read_csv('/content/kaggle/input/store-sales-time-series-forecasting/transactions.csv',
                                 usecols=['date', 'store_nbr', 'transactions'])

# Convert 'date' to datetime and downcast numeric columns
oil_data['date'] = pd.to_datetime(oil_data['date'])
holidays_data['date'] = pd.to_datetime(holidays_data['date'])
transactions_data['date'] = pd.to_datetime(transactions_data['date'])
transactions_data['store_nbr'] = transactions_data['store_nbr'].astype('category')


In [16]:
chunk_size = 100000  # Adjust chunk size based on RAM capacity
chunks = pd.read_csv('/content/kaggle/input/store-sales-time-series-forecasting/train.csv', chunksize=chunk_size)

for chunk in chunks:
    # Perform feature engineering on each chunk
    chunk['date'] = pd.to_datetime(chunk['date'])
    # Add other feature engineering steps here
    #...


In [ ]:
def rolling_mean_generator(data, window):
    for i in range(len(data) - window + 1):
        yield np.mean(data[i:i+window])

# Use generator for rolling mean calculation
train['rolling_mean_7'] = list(rolling_mean_generator(train['sales'].values, window=7))


In [ ]:
from memory_profiler import memory_usage

def process_data():
    # Your data processing code
    ...

mem_usage = memory_usage(process_data)
print(f"Memory usage: {max(mem_usage)} MB")


In [4]:
# Load and filter data
train = pd.read_csv('/content/kaggle/input/store-sales-time-series-forecasting/train.csv')
# train = train[train['date'] >= '2017-01-01']

# Downcast data types
for col in train.select_dtypes(include=['float64']).columns:
    train[col] = pd.to_numeric(train[col], downcast='float')
for col in train.select_dtypes(include=['int64']).columns:
    train[col] = pd.to_numeric(train[col], downcast='integer')

train['store_nbr'] = train['store_nbr'].astype('category')
train['family'] = train['family'].astype('category')
# Convert date to datetime after loading and filtering
train['date'] = pd.to_datetime(train['date']) # Convert the type of the date column after the dataframe has been created


def generate_all_time_series_features(
    df: pd.DataFrame,
    value_col: str,
    entity_col: str,
    time_cols: list,
    time_unit: str,
    lag_range: tuple = (1, 52),
    year_vicinity: int = 2,
    rolling_windows: list = [4, 12, 26, 52],
    max_zeros_windows: list = [4, 12, 26, 52]
) -> pd.DataFrame:
    """
    Generate various time series features for forecasting tasks.

    This function combines several feature generation methods:
    1. Last year's locality features
    2. Lag features
    3. Rolling mean features
    4. Cumulative sum features
    5. Maximum consecutive zeros features

    Parameters:
    - df: Input DataFrame containing time series data
    - value_col: Name of the column containing the target values
    - entity_col: Name of the column identifying unique entities (e.g., products)
    - time_cols: List of columns used for sorting and time-based calculations
    - time_unit: Time unit of the data ('day', 'week', or 'month')
    - lag_range: Tuple of (min_lag, max_lag) for generating lag features
    - year_vicinity: Number of time units to consider around the previous year's data
    - rolling_windows: List of window sizes for rolling mean calculations
    - max_zeros_windows: List of window sizes for calculating max consecutive zeros

    Returns:
    - DataFrame with additional feature columns
    """

    # Sort the DataFrame
    df = df.sort_values(by=time_cols)

    # 1. Generate last year's locality features
    cycle_size = {'month': 12, 'week': 52, 'day': 365}[time_unit]
    lags = range(cycle_size - year_vicinity, cycle_size + year_vicinity + 1)

    for lag in tqdm(lags, desc="Generating last year's locality features"):
        df[f'{value_col}_year_ago_lag{lag}'] = df.groupby(entity_col)[value_col].shift(lag)

    # 2. Generate lag features
    min_lag, max_lag = lag_range
    for lag in tqdm(range(min_lag, max_lag + 1), desc="Generating lag features"):
        df[f'{value_col}_lag{lag}'] = df.groupby(entity_col)[value_col].shift(lag)

    # Calculate lag ratios
    for lag in tqdm(range(min_lag, max_lag), desc="Generating lag ratios"):
        df[f'{value_col}_lag{lag}_ratio'] = df[f'{value_col}_lag{lag}'] / df[f'{value_col}_lag{lag+1}']
        df.loc[df[f'{value_col}_lag{lag}_ratio'] == np.inf, f'{value_col}_lag{lag}_ratio'] = 2
        df.loc[df[f'{value_col}_lag{lag}_ratio'].isna(), f'{value_col}_lag{lag}_ratio'] = 1

    # 3. Generate rolling mean features
    n_points = {'week': 4, 'day': 30, 'month': 1}[time_unit]
    cycle = {'week': 52, 'day': 365, 'month': 12}[time_unit]

    tqdm.pandas(desc="Generating rolling mean features")
    df[f'{value_col}_roll_mean'] = df.groupby(entity_col)[value_col].transform(
        lambda x: x.rolling(window=n_points).mean()
    )
    df[f'{value_col}_roll_mean_year_ago'] = df.groupby(entity_col)[f'{value_col}_roll_mean'].shift(cycle)
    df['roll_mean_ratio'] = df[f'{value_col}_roll_mean'] / df[f'{value_col}_roll_mean_year_ago']

    df.loc[df['roll_mean_ratio'] == np.inf, 'roll_mean_ratio'] = np.nan
    df.loc[(df[f'{value_col}_roll_mean'] == 0) & (df[f'{value_col}_roll_mean_year_ago'] == 0), 'roll_mean_ratio'] = 1

    # 4. Generate cumulative sum features
    df['year'] = pd.to_datetime(df[time_cols[0]]).dt.year
    tqdm.pandas(desc="Generating cumulative sum features")
    df[f'{value_col}_cumsum'] = df.groupby([entity_col, 'year'])[value_col].transform('cumsum')
    df[f'{value_col}_cumsum_last_year'] = df.groupby(entity_col)[f'{value_col}_cumsum'].shift(cycle)
    df[f'{value_col}_total_last_year'] = df.groupby([entity_col, 'year'])[value_col].transform('sum').shift(cycle)

    df['cumsum_ratio'] = df[f'{value_col}_cumsum'] / df[f'{value_col}_cumsum_last_year']
    df['cumsum_total_ratio'] = df[f'{value_col}_cumsum'] / df[f'{value_col}_total_last_year']

    # Handle edge cases
    for col in ['cumsum_ratio', 'cumsum_total_ratio']:
        df.loc[(df[f'{value_col}_cumsum'] == 0) & (df[f'{value_col}_cumsum_last_year'] == 0), col] = 1
        df.loc[(df[f'{value_col}_cumsum'] > 0) & (df[f'{value_col}_cumsum_last_year'] == 0), col] = 2

    df = df.drop([f'{value_col}_cumsum', f'{value_col}_cumsum_last_year', f'{value_col}_total_last_year', 'year'], axis=1)

    # 5. Generate maximum consecutive zeros features
    df['is_zero'] = (df[value_col] == 0).astype(int)
    tqdm.pandas(desc="Generating consecutive zeros features")
    df['consec_zeros'] = df.groupby(entity_col)['is_zero'].transform(
        lambda x: x.groupby((x != x.shift()).cumsum()).cumsum()
    )

    for window in tqdm(max_zeros_windows, desc="Calculating max consecutive zeros"):
        df[f'max_consec_zeros_w{window}'] = df.groupby(entity_col)['consec_zeros'].transform(
            lambda x: x.rolling(window=window).max()
        )

    df = df.drop(['is_zero', 'consec_zeros'], axis=1)

    return df

# Generate selective features
train = generate_all_time_series_features(
    df=train,
    value_col='sales',
    entity_col='family',
    time_cols=['date'],
    time_unit='day',
    lag_range=(1, 14),
    year_vicinity=3,
    rolling_windows=[7, 14],
    max_zeros_windows=[7, 14]
)

# Merge additional data
train = train.merge(oil_data, how='left', on='date')
train = train.merge(holidays_data, how='left', on='date')
train = train.merge(transactions_data, how='left', on=['date', 'store_nbr'])


Generating last year's locality features:   0%|          | 0/7 [00:00<?, ?it/s]<ipython-input-4-49d2f1c016a3>:61: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df[f'{value_col}_year_ago_lag{lag}'] = df.groupby(entity_col)[value_col].shift(lag)
<ipython-input-4-49d2f1c016a3>:61: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df[f'{value_col}_year_ago_lag{lag}'] = df.groupby(entity_col)[value_col].shift(lag)
Generating last year's locality features:  29%|██▊       | 2/7 [00:00<00:00, 12.94it/s]<ipython-input-4-49d2f1c016a3>:61: FutureWarning: The default of observed=False is deprecated and will be changed to Tru

In [5]:
train['store_nbr'] = train['store_nbr'].astype(str)
train['family'] = train['family'].astype(str)
# test['store_nbr'] = test['store_nbr'].astype(str)
# test['family'] = test['family'].astype(str)
train['item_id'] = train['store_nbr'] + '_' + train['family']
# test['item_id'] = test['store_nbr'] + '_' + test['family']
train = train.rename(columns={'date':'timestamp'})
# test = test.rename(columns={'date':'timestamp'})
# setting a temporary index as autogluon requires
train.set_index(['item_id','timestamp'], inplace=True)
# test.set_index(['item_id','timestamp'], inplace=True)
train = train.rename(columns={'sales':'target'})
# train_copy = train.copy()

In [ ]:
# import pandas as pd
import numpy as np
import gc  # For garbage collection

# # Replace inf values and check memory usage
train.replace([np.inf, -np.inf], np.nan, inplace=True)

# Ensure optimal data types for large datasets
for col in train.select_dtypes(include=['float64']).columns:
    train[col] = pd.to_numeric(train[col], downcast='float')
for col in train.select_dtypes(include=['int64']).columns:
    train[col] = pd.to_numeric(train[col], downcast='integer')


train_end_date = '2016-12-31'
val_end_date = '2017-06-30'
test_start_date = '2017-07-01'

# Define split dates and subset the data
# train_end_date = '2017-03-31'
# val_end_date = '2017-06-30'
# test_start_date = '2017-07-01'


# 2013-01-01 to 2017-03-31: 4 years
# 2017-03-31 to 2017-06-30: 3 months
# 2017-07-01 to 2017-08-15 : 1.5 month

train_data = train.loc[train.index.get_level_values('timestamp') < train_end_date]
val_data = train.loc[(train.index.get_level_values('timestamp') >= train_end_date) &
                     (train.index.get_level_values('timestamp') <= val_end_date)]
test_data = train.loc[train.index.get_level_values('timestamp') >= test_start_date]

# Clear unused variable and force garbage collection


# Setup TimeSeriesPredictor
from autogluon.timeseries import TimeSeriesPredictor
save_path = '/content/kaggle/working/models'
predictor = TimeSeriesPredictor(freq='D', prediction_length=7, eval_metric='MASE', path=save_path)

# Fit the model
predictor.fit(
    train_data=train_data.reset_index(),  # Reset index to convert multi-index to columns
    tuning_data=val_data.reset_index(),

)

# Generate leaderboard
leaderboard = predictor.leaderboard(test_data.reset_index(), silent=True)
print(leaderboard)

# Save leaderboard to CSV
df_val_results = pd.DataFrame(leaderboard)
df_val_results.to_csv('test_results_with_FE.csv', index=False)

Beginning AutoGluon training...
AutoGluon will save models to '/content/kaggle/working/models'
=================== System Info ===================
AutoGluon Version:  1.2
Python Version:     3.10.12
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP PREEMPT_DYNAMIC Thu Jun 27 21:05:47 UTC 2024
CPU Count:          2
GPU Count:          0
Memory Avail:       8.10 GB / 12.67 GB (63.9%)
Disk Space Avail:   71.85 GB / 107.72 GB (66.7%)

Fitting with arguments:
{'enable_ensemble': True,
 'eval_metric': MASE,
 'freq': 'D',
 'hyperparameters': 'default',
 'known_covariates_names': [],
 'num_val_windows': 1,
 'prediction_length': 7,
 'quantile_levels': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9],
 'random_seed': 123,
 'refit_every_n_windows': 1,
 'refit_full': False,
 'skip_model_selection': False,
 'target': 'target',
 'verbosity': 2}

train_data with frequency 'None' has been resampled to frequency 'D'.
Provided train_data has 2601720 rows (NaN fraction=0.3%), 1

In [ ]:
train_data

# Trial 2
Just below cell Can be removed

In [ ]:
#import pandas as pd
import numpy as np
import gc  # For garbage collection

# Replace inf values and check memory usage
train.replace([np.inf, -np.inf], np.nan, inplace=True)

# Ensure optimal data types for large datasets
for col in train.select_dtypes(include=['float64']).columns:
    train[col] = pd.to_numeric(train[col], downcast='float')
for col in train.select_dtypes(include=['int64']).columns:
    train[col] = pd.to_numeric(train[col], downcast='integer')


train_end_date = '2016-12-31'
val_end_date = '2017-06-30'
test_start_date = '2017-07-01'


# Define split dates and subset the data
# train_end_date = '2017-03-31'
# val_end_date = '2017-06-30'
# test_start_date = '2017-07-01'


# 2013-01-01 to 2017-03-31: 4 years
# 2017-03-31 to 2017-06-30: 3 months
# 2017-07-01 to 2017-08-15 : 1.5 month

train_data = train.loc[train.index.get_level_values('timestamp') < train_end_date]
val_data = train.loc[(train.index.get_level_values('timestamp') >= train_end_date) &
                     (train.index.get_level_values('timestamp') <= val_end_date)]
test_data = train.loc[train.index.get_level_values('timestamp') >= test_start_date]

# Clear unused variable and force garbage collection


# Setup TimeSeriesPredictor
from autogluon.timeseries import TimeSeriesPredictor
save_path = '/content/kaggle/working/models'
predictor = TimeSeriesPredictor(freq='D', prediction_length=7, eval_metric='MASE', path=save_path)

# Fit the model
predictor.fit(
    train_data=train_data.reset_index(),  # Reset index to convert multi-index to columns
    tuning_data=val_data.reset_index(),

)

# Generate leaderboard
leaderboard = predictor.leaderboard(test_data.reset_index(), silent=True)
print(leaderboard)

# Save leaderboard to CSV
df_val_results = pd.DataFrame(leaderboard)
df_val_results.to_csv('test_results_with_FE.csv', index=False)

In [ ]:
predictor.info()

In [ ]:
model_names = predictor.model_names()
for i in model_names:print(i)

In [ ]:
import os
import pandas as pd
import joblib
from autogluon.timeseries import TimeSeriesDataFrame

models_dir = '/content/kaggle/working/models/models'

predictions = {}

# Convert test_data into the correct format (TimeSeriesDataFrame)
df = TimeSeriesDataFrame(test_data.reset_index())

# Sort the DataFrame to avoid the warning
df = df.sort_index()

# Optionally, fill missing values
df = df.fill_missing_values()

for model_folder in os.listdir(models_dir):
    model_path = os.path.join(models_dir, model_folder, 'model.pkl')

    if os.path.exists(model_path):
        model = joblib.load(model_path)

        if model is not None:
            try:
                preds = model.predict(df)
                predictions[model_folder] = preds
            except Exception as e:
                print(f"Error with model {model_folder}: {e}")
        else:
            print(f"Model {model_folder} could not be loaded.")

predictions["original"] = test_data["target"]

predictions_df = pd.DataFrame(predictions)
predictions_df.to_csv('/content/kaggle/working/test_predictions.csv', index=False)

print("Predictions saved to '/content/kaggle/working/test_predictions.csv'")


In [ ]:
for model_folder in os.listdir(models_dir):
    print(model_folder)

# Again trial 2 came

In [ ]:
import pandas as pd
import numpy as np
import gc  # For garbage collection

# Replace inf values and check memory usage
train.replace([np.inf, -np.inf], np.nan, inplace=True)

# Ensure optimal data types for large datasets
for col in train.select_dtypes(include=['float64']).columns:
    train[col] = pd.to_numeric(train[col], downcast='float')
for col in train.select_dtypes(include=['int64']).columns:
    train[col] = pd.to_numeric(train[col], downcast='integer')

# Define split dates and subset the data
train_end_date = '2016-10-01'  # 3yrs 9 months
val_end_date = '2017-06-30'    # 6 months
test_start_date = '2017-07-01'

train_data = train.loc[train.index.get_level_values('timestamp') < train_end_date]
val_data = train.loc[(train.index.get_level_values('timestamp') >= train_end_date) &
                     (train.index.get_level_values('timestamp') <= val_end_date)]
test_data = train.loc[train.index.get_level_values('timestamp') >= test_start_date]

# Clear unused variable and force garbage collection
del train
gc.collect()

# Setup TimeSeriesPredictor
from autogluon.timeseries import TimeSeriesPredictor
save_path = '/content/kaggle/working/models'
predictor = TimeSeriesPredictor(freq='D', prediction_length=7, eval_metric='MASE', path=save_path)

# Fit the model
predictor.fit(
    train_data=train_data.reset_index(),  # Reset index to convert multi-index to columns
    tuning_data=val_data.reset_index(),
    #time_limit=12800
)

# Generate leaderboard
leaderboard = predictor.leaderboard(test_data.reset_index(), silent=True)
print(leaderboard)

# Save leaderboard to CSV
df_val_results = pd.DataFrame(leaderboard)
df_val_results.to_csv('test_results_with_FE_1.csv', index=False)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Generate predictions with quantiles
predictions = predictor.predict(val_data.reset_index())

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import gc  # For garbage collection
# Set up figure and axes for plotting
plt.figure(figsize=(14, 8))
val_data = val_data.reset_index()
test_data = test_data.reset_index()
predictions = predictions.reset_index()


In [ ]:
item_id = '4_EGGS'
selected_val = val_data[(val_data['item_id'] == item_id) & (val_data['timestamp'] > '2017-06-15')]
selected_test = test_data[(test_data['item_id'] == item_id) & (test_data['timestamp'] < '2017-07-08')]
selected_pred = predictions[predictions['item_id'] == item_id]

plt.plot(selected_val['timestamp'], selected_val['target'], label=f'{item_id} - Validation Actuals')
plt.plot(selected_test['timestamp'], selected_test['target'], label=f'{item_id} - Test Actuals')
plt.plot(selected_pred['timestamp'], selected_pred['mean'], label=f'{item_id} - Prediction Mean', linestyle='--')
# Optional: add quantile bounds if available
plt.fill_between(selected_pred['timestamp'],
                 selected_pred['0.1'],
                 selected_pred['0.9'],
                 alpha=0.2, label=f'{item_id} - Prediction 10-90% Quantiles')
plt.xlabel('Date')
plt.ylabel('Target Value')
plt.title(f'Predictions vs Actuals for {item_id}')
plt.legend()
plt.show()

In [ ]:
item_id = '2_EGGS'
selected_val = val_data[(val_data['item_id'] == item_id) & (val_data['timestamp'] > '2017-06-15')]
selected_test = test_data[(test_data['item_id'] == item_id) & (test_data['timestamp'] < '2017-07-08')]
selected_pred = predictions[predictions['item_id'] == item_id]

# Concatenate validation and test data for continuous actuals line
combined_actuals = pd.concat([selected_val, selected_test])

# Plot combined actuals for a smoother transition
plt.plot(combined_actuals['timestamp'], combined_actuals['target'], label=f'{item_id} - Actuals', color='orange')
plt.plot(selected_pred['timestamp'], selected_pred['mean'], label=f'{item_id} - Prediction Mean', linestyle='--')

# Optional: add quantile bounds if available
plt.fill_between(selected_pred['timestamp'],
                 selected_pred['0.1'],
                 selected_pred['0.9'],
                 alpha=0.2, label=f'{item_id} - Prediction 10-90% Quantiles')
plt.xlabel('Date')
plt.ylabel('Target Value')
plt.title(f'Predictions vs Actuals for {item_id}')
plt.legend()
plt.show()

In [ ]:
item_id = '3_EGGS'
selected_val = val_data[(val_data['item_id'] == item_id) & (val_data['timestamp'] > '2017-06-15')]
selected_test = test_data[(test_data['item_id'] == item_id) & (test_data['timestamp'] < '2017-07-08')]
selected_pred = predictions[predictions['item_id'] == item_id]

# Concatenate validation and test data for continuous actuals line
combined_actuals = pd.concat([selected_val, selected_test])

# Plot combined actuals for a smoother transition
plt.plot(combined_actuals['timestamp'], combined_actuals['target'], label=f'{item_id} - Actuals', color='orange')
plt.plot(selected_pred['timestamp'], selected_pred['mean'], label=f'{item_id} - Prediction Mean', linestyle='--')

# Optional: add quantile bounds if available
plt.fill_between(selected_pred['timestamp'],
                 selected_pred['0.1'],
                 selected_pred['0.9'],
                 alpha=0.2, label=f'{item_id} - Prediction 10-90% Quantiles')
plt.xlabel('Date')
plt.ylabel('Target Value')
plt.title(f'Predictions vs Actuals for {item_id}')
plt.legend()
plt.show()


In [ ]:
item_id = '4_EGGS'
selected_val = val_data[(val_data['item_id'] == item_id) & (val_data['timestamp'] > '2017-06-15')]
selected_test = test_data[(test_data['item_id'] == item_id) & (test_data['timestamp'] < '2017-07-08')]
selected_pred = predictions[predictions['item_id'] == item_id]

# Concatenate validation and test data for continuous actuals line
combined_actuals = pd.concat([selected_val, selected_test])

# Plot combined actuals for a smoother transition
plt.plot(combined_actuals['timestamp'], combined_actuals['target'], label=f'{item_id} - Actuals', color='orange')
plt.plot(selected_pred['timestamp'], selected_pred['mean'], label=f'{item_id} - Prediction Mean', linestyle='--')

# Optional: add quantile bounds if available
plt.fill_between(selected_pred['timestamp'],
                 selected_pred['0.1'],
                 selected_pred['0.9'],
                 alpha=0.2, label=f'{item_id} - Prediction 10-90% Quantiles')
plt.xlabel('Date')
plt.ylabel('Target Value')
plt.title(f'Predictions vs Actuals for {item_id}')
plt.legend()
plt.show()

In [ ]:
item_id = '6_EGGS'
selected_val = val_data[(val_data['item_id'] == item_id) & (val_data['timestamp'] > '2017-06-15')]
selected_test = test_data[(test_data['item_id'] == item_id) & (test_data['timestamp'] < '2017-07-08')]
selected_pred = predictions[predictions['item_id'] == item_id]

# Concatenate validation and test data for continuous actuals line
combined_actuals = pd.concat([selected_val, selected_test])

# Plot combined actuals for a smoother transition
plt.plot(combined_actuals['timestamp'], combined_actuals['target'], label=f'{item_id} - Actuals', color='orange')
plt.plot(selected_pred['timestamp'], selected_pred['mean'], label=f'{item_id} - Prediction Mean', linestyle='--')

# Optional: add quantile bounds if available
plt.fill_between(selected_pred['timestamp'],
                 selected_pred['0.1'],
                 selected_pred['0.9'],
                 alpha=0.2, label=f'{item_id} - Prediction 10-90% Quantiles')
plt.xlabel('Date')
plt.ylabel('Target Value')
plt.title(f'Predictions vs Actuals for {item_id}')
plt.legend()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import os

# Directory where you want to save the plots
output_dir = '/content/kaggle/working/plots'
os.makedirs(output_dir, exist_ok=True)  # Create the directory if it doesn't exist

unique_items = val_data['item_id'].unique()

# Loop over each unique item_id
for i,item_id in enumerate(unique_items):
    try:
        # Filter data for the specific item
        selected_val = val_data[(val_data['item_id'] == item_id) & (val_data['timestamp'] > '2017-06-15')]
        selected_test = test_data[(test_data['item_id'] == item_id) & (test_data['timestamp'] < '2017-07-08')]
        selected_pred = predictions[predictions['item_id'] == item_id]

        # Combine validation and test actuals for smooth plotting
        combined_actuals = pd.concat([selected_val, selected_test])

        # Plot the data
        plt.figure(figsize=(10, 6))
        plt.plot(combined_actuals['timestamp'], combined_actuals['target'], label=f'{item_id} - Actuals', color='orange')
        plt.plot(selected_pred['timestamp'], selected_pred['mean'], label=f'{item_id} - Prediction Mean', linestyle='--')

        # Optional: add quantile bounds if available
        plt.fill_between(selected_pred['timestamp'],
                         selected_pred['0.1'],
                         selected_pred['0.9'],
                         alpha=0.2, label=f'{item_id} - Prediction 10-90% Quantiles')

        # Set labels, title, and legend
        plt.xlabel('Date')
        plt.ylabel('Target Value')
        plt.title(f'Predictions vs Actuals for {item_id}')
        plt.legend()

        # Save the figure to the output directory
        plt.savefig(f'{output_dir}/Predictions_vs_Actuals_{item_id}.png')
        plt.close()  # Close the figure to save memory
    except:
        continue

print("Charts have been saved for all items.")

In [ ]:
import shutil

# Path to the folder you want to zip
folder_path = '/content/kaggle/working/plots'
zip_path = '/content/kaggle/working/plots.zip'

# Zip the folder
shutil.make_archive(zip_path.replace('.zip', ''), 'zip', folder_path)

In [ ]:
# item_data = full_data[full_data['item_id'] == '1_AUTOMOTIVE']
plt.figure(figsize=(12, 6))
plt.plot(item_data['timestamp'], item_data['mean'], label='Mean Prediction', color='blue')
plt.plot(item_data['timestamp'], item_data['target'], label='Actual (Train Data)', color='black', linestyle='--')
plt.fill_between(
    item_data['timestamp'],
    item_data['0.1'],  # 10th percentile
    item_data['0.9'],  # 90th percentile
    color='blue',
    alpha=0.3,
    label='10th to 90th Percentile'
)

# Add additional lines or fill_between if you want to plot more quantiles, such as 20th to 80th
plt.xlabel("Date")
plt.ylabel("Value")
plt.title(f"Actual vs Predicted with Percentiles for Item {'1_AUTOMOTIVE'}")
plt.legend()
plt.show()

In [ ]:
predictions

In [ ]:
# Plotting
plt.figure(figsize=(12, 6))

# Assuming 'predictions' has a 'mean' column and quantile columns for '0.1', '0.5', '0.9'
for item_id in predictions['item_id'].unique():  # Adjust based on how your data is structured
    item_predictions = predictions[predictions['item_id'] == item_id]

    plt.plot(item_predictions['timestamp'], item_predictions['0.5'], label=f"{item_id} - 50th Percentile", color='blue')
    plt.fill_between(
        item_predictions['timestamp'],
        item_predictions['0.1'],
        item_predictions['0.9'],
        color='blue',
        alpha=0.3,
        label=f"{item_id} - 10th to 90th Percentile"
    )

# Add labels and legend
plt.xlabel("Date")
plt.ylabel("Predicted Value")
plt.title("Predictions with Percentiles")
plt.legend()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import gc  # For garbage collection

# Replace inf values and check memory usage
train.replace([np.inf, -np.inf], np.nan, inplace=True)

# Ensure optimal data types for large datasets
for col in train.select_dtypes(include=['float64']).columns:
    train[col] = pd.to_numeric(train[col], downcast='float')
for col in train.select_dtypes(include=['int64']).columns:
    train[col] = pd.to_numeric(train[col], downcast='integer')

# Define split dates and subset the data
# train_end_date = '2017-01-01'
train_end_date = '2017-06-30'
test_start_date = '2017-07-01'

train_data = train.loc[train.index.get_level_values('timestamp') < train_end_date]
# val_data = train.loc[(train.index.get_level_values('timestamp') >= train_end_date) &
#                      (train.index.get_level_values('timestamp') <= val_end_date)]
test_data = train.loc[train.index.get_level_values('timestamp') >= test_start_date]

# Clear unused variable and force garbage collection
del train
gc.collect()

# Setup TimeSeriesPredictor
from autogluon.timeseries import TimeSeriesPredictor
save_path = 'autogluon_models'
predictor = TimeSeriesPredictor(freq='D', prediction_length=7, eval_metric='MASE', path=save_path)

# Fit the model
predictor.fit(
    train_data=train_data.reset_index(),  # Reset index to convert multi-index to columns
    time_limit=12800
)

# Generate leaderboard
leaderboard = predictor.leaderboard(test_data.reset_index(), silent=True)
print(leaderboard)

# Save leaderboard to CSV
df_val_results = pd.DataFrame(leaderboard)
df_val_results.to_csv('test_results_with_FE.csv', index=False)

In [ ]:
leaderboard

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
from autogluon.timeseries import TimeSeriesPredictor, TimeSeriesDataFrame
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Constants
TEST = 180  # 180 days for test set
PDT = 7     # Weekly prediction length
train_end_date = '2016-08-15'  # End of the training period
val_end_date = '2017-02-15'    # End of the validation period

# Step 1: Data Loading and Preprocessing
df = train_copy.copy().reset_index()  # Start with the dataset
df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')  # Convert to datetime
df.set_index('timestamp', inplace=True)

# # Step 2: Keep only relevant columns
# columns_to_keep = ['id', 'store_nbr', 'family', 'target', 'onpromotion', 'dcoilwtico',
#                    'type', 'locale', 'locale_name', 'transferred', 'transactions']
# df = df[columns_to_keep]

# Step 3: Memory optimization
df['store_nbr'] = df['store_nbr'].astype('int32')
df['family'] = df['family'].astype('category')

# Step 4: Aggregate data by unique timestamp, store, and family
df_aggregated = df.groupby(['timestamp', 'store_nbr', 'family']).agg({
    'target': 'sum',
    'onpromotion': 'sum',
    'dcoilwtico': 'mean',
    'transactions': 'mean'
}).reset_index()

# Step 5: Create unique item_id for each store-family combination
df_aggregated['item_id'] = df_aggregated['family'].astype(str) + '_' + df_aggregated['store_nbr'].astype(str)
df_aggregated.set_index('timestamp', inplace=True)

# Step 6: Splitting Data into Train, Validation, and Test Sets
train_data = df_aggregated.loc[df_aggregated.index < train_end_date]
val_data = df_aggregated.loc[(df_aggregated.index >= train_end_date) & (df_aggregated.index < val_end_date)]
# test_data = df_aggregated.loc[df_aggregated.index >= val_end_date]
test_data = df_aggregated.loc[df_aggregated.index >= '2017-02-16']


# Step 7: Convert to AutoGluon’s TimeSeriesDataFrame Format
train_data_tsdf = TimeSeriesDataFrame.from_data_frame(train_data.reset_index(), id_column="item_id", timestamp_column="timestamp")
val_data_tsdf = TimeSeriesDataFrame.from_data_frame(val_data.reset_index(), id_column="item_id", timestamp_column="timestamp")
test_data_tsdf = TimeSeriesDataFrame.from_data_frame(test_data.reset_index(), id_column="item_id", timestamp_column="timestamp")

# Step 8: Train AutoGluon TimeSeriesPredictor
predictor = TimeSeriesPredictor(
    freq='D',
    prediction_length=PDT,
    eval_metric='MASE',
    path='autogluon_models'
)

predictor.fit(
    train_data=train_data_tsdf,
    tuning_data=val_data_tsdf,
    time_limit=10800
)

# Get the leaderboard for all trained models
leaderboard = predictor.leaderboard(test_data_tsdf, silent=False)  # Leaderboard for validation data
print("Model Leaderboard on test Set:")
print(leaderboard)
df_leaderboard = pd.DataFrame(leaderboard)
df_leaderboard.to_csv('without_feature_engineering.csv')


In [ ]:
import pandas as pd
import numpy as np
from gluonts.dataset.pandas import PandasDataset
from gluonts.dataset.split import split
from uni2ts.model.moirai import MoiraiForecast, MoiraiModule

# Define constants
TEST = 180  # 180 days for test
PDT = 7     # Weekly prediction
CTX = 30    # Context length
PSZ = 7     # Patch size for Moirai
BSZ = 32    # Batch size

# Load and preprocess data
df = train_copy.copy().reset_index()
df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
df.set_index('timestamp', inplace=True)

# Filter necessary columns and aggregate for unique timestamps
columns_to_keep = ['id', 'store_nbr', 'family', 'target', 'onpromotion', 'dcoilwtico', 'type', 'locale', 'locale_name', 'transferred', 'transactions']
df = df[columns_to_keep]
df['store_nbr'] = df['store_nbr'].astype('int32')
df['family'] = df['family'].astype('category')

df_aggregated = df.groupby(['timestamp', 'store_nbr', 'family']).agg({
    'target': 'sum',
    'onpromotion': 'sum',
    'dcoilwtico': 'mean',
    'transactions': 'mean'
}).reset_index()

df_aggregated['item_id'] = df_aggregated['family'].astype(str) + '_' + df_aggregated['store_nbr'].astype(str)
df_aggregated.set_index('timestamp', inplace=True)

# Convert to GluonTS dataset
ds = PandasDataset.from_long_dataframe(df_aggregated.reset_index(), target="target", item_id="item_id")

# Split into train/test
train, test_template = split(ds, offset=-TEST)
test_data = test_template.generate_instances(
    prediction_length=PDT,
    windows=TEST // PDT,
    distance=PDT,
)

# Model setup
model = MoiraiForecast(
    module=MoiraiModule.from_pretrained("Salesforce/moirai-1.0-R-small"),
    prediction_length=PDT,
    context_length=CTX,
    patch_size=PSZ,
    num_samples=100,
    target_dim=1,
    feat_dynamic_real_dim=ds.num_feat_dynamic_real,
    past_feat_dynamic_real_dim=ds.num_past_feat_dynamic_real,
)

# Generate forecasts
predictor = model.create_predictor(batch_size=BSZ)
forecasts_list = list(predictor.predict(test_data.input))

# Scale for MASE calculation
scale = np.mean([np.abs(y["target"]).mean() for y in test_data.label])

# Manual MAE calculation
overall_results = []

for input_data, label_data, forecast_data in zip(test_data.input, test_data.label, forecasts_list):
    forecast_values = forecast_data.mean(axis=0) if isinstance(forecast_data, list) else forecast_data.samples.mean(axis=0)

    if len(forecast_values) > 0:
        # Ensure non-null forecast and target values
        target_values = label_data["target"]
        mask = ~np.isnan(forecast_values) & ~np.isnan(target_values)  # Filter out NaNs
        valid_forecast_values = forecast_values[mask]
        valid_target_values = target_values[mask]

        # Calculate MAE for this specific instance
        if len(valid_forecast_values) > 0:
            mae_instance = np.mean(np.abs(valid_forecast_values - valid_target_values))

            item_id = label_data["item_id"]
            family_code, store_code = item_id.split('_')

            overall_results.append({
                'family': family_code,
                'store': store_code,
                'mae': mae_instance,
                'forecast': valid_forecast_values.tolist()
            })

# Convert to DataFrame and compute overall metrics
overall_results_df = pd.DataFrame(overall_results)
overall_mae = overall_results_df['mae'].mean() if not overall_results_df.empty else np.nan
mase = overall_mae / scale if scale != 0 else np.nan

# Display final summary
final_summary = {
    'overall_mae': overall_mae,
    'overall_mase': mase,
}

print(final_summary)

In [ ]:
forecast_values

In [ ]:
target_values

In [ ]:
for model_name in leaderboard['model']:
    print(f"\nEvaluating Model: {model_name}")
    # Evaluate MASE, MAE, MSE for each model on the test data
    test_scores = predictor.evaluate(test_data_tsdf, model=model_name)
    print(f"Test MASE for {model_name}: {test_scores['MASE']}")
    print(f"Test MAE for {model_name}: {test_scores['MAE']}")
    print(f"Test MSE for {model_name}: {test_scores['MSE']}")

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
from autogluon.timeseries import TimeSeriesPredictor, TimeSeriesDataFrame
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Constants
TEST = 180  # 180 days for test set
PDT = 7     # Weekly prediction length
train_end_date = '2016-07-31'  # End of the training period
val_end_date = '2017-01-01'    # End of the validation period

# Step 1: Data Loading and Preprocessing
df = train_copy.copy().reset_index()  # Start with the dataset
df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')  # Convert to datetime
df.set_index('timestamp', inplace=True)

# Step 2: Keep only relevant columns
columns_to_keep = ['id', 'store_nbr', 'family', 'target', 'onpromotion', 'dcoilwtico',
                   'type', 'locale', 'locale_name', 'transferred', 'transactions']
df = df[columns_to_keep]

# Step 3: Memory optimization
df['store_nbr'] = df['store_nbr'].astype('int32')
df['family'] = df['family'].astype('category')

# Step 4: Aggregate data by unique timestamp, store, and family
df_aggregated = df.groupby(['timestamp', 'store_nbr', 'family']).agg({
    'target': 'sum',
    'onpromotion': 'sum',
    'dcoilwtico': 'mean',
    'transactions': 'mean'
}).reset_index()

# Step 5: Create unique item_id for each store-family combination
df_aggregated['item_id'] = df_aggregated['family'].astype(str) + '_' + df_aggregated['store_nbr'].astype(str)
df_aggregated.set_index('timestamp', inplace=True)

# Step 6: Splitting Data into Train, Validation, and Test Sets
train_data = df_aggregated.loc[df_aggregated.index < train_end_date]
val_data = df_aggregated.loc[(df_aggregated.index >= train_end_date) & (df_aggregated.index < val_end_date)]
test_data = df_aggregated.loc[df_aggregated.index >= val_end_date]

# Step 7: Convert to AutoGluon’s TimeSeriesDataFrame Format
train_data = train_data.reset_index()
val_data = val_data.reset_index()
test_data = test_data.reset_index()

train_data_tsdf = TimeSeriesDataFrame.from_data_frame(train_data, id_column="item_id", timestamp_column="timestamp")
val_data_tsdf = TimeSeriesDataFrame.from_data_frame(val_data, id_column="item_id", timestamp_column="timestamp")
test_data_tsdf = TimeSeriesDataFrame.from_data_frame(test_data, id_column="item_id", timestamp_column="timestamp")


In [ ]:
test_data_tsdf

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# Step 2: Initialize lists to store metrics
mae_list = []
mse_list = []
rmse_list = []

# Ensure 'item_id' exists in the predictions index
if 'item_id' not in predictions.index.names:
    raise KeyError("The 'item_id' is not found in the index of the predictions DataFrame.")

# List the unique item_id values in test_data_tsdf and predictions
print("Unique item_id in test_data_tsdf:", test_data_tsdf.index.get_level_values('item_id').unique())
print("Unique item_id in predictions:", predictions.index.get_level_values('item_id').unique())

# For each item_id, try to retrieve actual values and check if it's non-empty
for item_id in predictions.index.get_level_values('item_id').unique():
    # Retrieve actual values
    actual_values = test_data_tsdf.loc[test_data_tsdf.index.get_level_values('item_id') == item_id, 'target']
    print(f"Actual values for item_id {item_id}: {actual_values}")

    if actual_values.empty:
        print(f"No actual values found for item_id {item_id}")

    # Retrieve predicted values for the current item_id
    predicted_values = predictions.loc[predictions.index.get_level_values('item_id') == item_id, 'mean']  # 'mean' if that's the column name

    # Print predicted values to check if they are populated
    print(f"Predicted values for item_id {item_id}:\n", predicted_values)

    if predicted_values.empty:
        print(f"No predicted values found for item_id {item_id}")


    # Align values directly and print results to debug
    actual_values, predicted_values = actual_values.align(predicted_values, join='inner')
    print("Aligned actual values:\n", actual_values)
    print("Aligned predicted values:\n", predicted_values)

# Check if alignment results in empty lists
if actual_values.empty or predicted_values.empty:
    print("Alignment resulted in empty values for either actual or predicted.")

    # Compute metrics for this item
    mae = mean_absolute_error(actual_values, predicted_values)
    mse = mean_squared_error(actual_values, predicted_values)
    rmse = np.sqrt(mse)

    # Append the results
    mae_list.append(mae)
    mse_list.append(mse)
    rmse_list.append(rmse)

# Step 4: Create a DataFrame to store metrics for each item
metrics_df = pd.DataFrame({
    'item_id': predictions.index.get_level_values('item_id').unique(),
    'MAE': mae_list,
    'MSE': mse_list,
    'RMSE': rmse_list
})

# Step 5: Calculate overall metrics (mean across all items)
overall_metrics = {
    'overall_MAE': np.mean(mae_list),
    'overall_MSE': np.mean(mse_list),
    'overall_RMSE': np.mean(rmse_list),
}

# Print individual item metrics and overall metrics
print("Metrics by item:\n", metrics_df)
print("\nOverall Metrics:\n", overall_metrics)

# Optionally, save metrics to CSV
metrics_df.to_csv('test_metrics_by_item.csv', index=False)
print("Metrics saved to 'test_metrics_by_item.csv'")

In [ ]:
print("done")